Final project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os
import json
from PIL import Image

DATA_DIR = "/content/drive/MyDrive/pixels_to_predictions"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Mounted at /content/drive
Train: (3109, 15)
Val: (1048, 15)
Test: (1008, 13)


In [2]:
# ── 0. Install libraries ──────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.19.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.9 MB/s eta 0:00:00


In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms as T
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("/content/drive/MyDrive/pixels_to_predictions")
MODEL_ID  = "HuggingFaceTB/SmolVLM-500M-Instruct"


IMG_SIZE = 384

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
torch.set_float32_matmul_precision('high')

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [4]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [5]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    # subject + grade
    subject = row.get("subject", "")
    grade   = row.get("grade", "")
    if pd.notna(subject) and str(subject).strip():
        meta = str(subject).strip()
        if pd.notna(grade) and str(grade).strip():
            meta += f" ({str(grade).strip()})"
        context_parts.append(f"[{meta}]")


    cap = caption_cache.get(row["image_path"], "").strip()
    if cap:
        context_parts.append(f"Image: {cap}")

    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "The correct answer is:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt



In [6]:
# ── 2c. Dataset — with training augmentation [Tip 8] ─────────────────────────

# Light augmentation applied only during training.
# Brightness/contrast jitter helps generalize across different diagram styles.
# No geometric transforms (flip/rotation) since orientation matters for diagrams.
train_aug = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2),   # [Tip 8]
])

class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = IMG_SIZE,
                 is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        # [Tip 7] resize to larger size so model can read axis labels / map text
        img = img.resize((self.img_size, self.img_size), Image.LANCZOS)
        if self.is_train:
            img = train_aug(img)   # [Tip 8] apply augmentation
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, is_train=False)

In [7]:
# ── 3a. Load SmolVLM model ────────────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    model.to(device)
model.eval()
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded.


In [8]:
from tqdm.auto import tqdm

CAPTION_CACHE = DATA_DIR / "image_captions_v2_cleaned.json"
CAPTION_RAW   = DATA_DIR / "image_captions_v2_raw.json"
CAPTION_BATCH = 32

CAPTION_MESSAGES = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": (
                "This is a science question image. "
                "Describe what you see: any diagrams, charts, labels, "
                "numbers, arrows, or text visible. "
                "Be specific and concise (1-2 sentences)."
            )},
        ],
    }
]

@torch.inference_mode()
def generate_captions_batch(images):
    prompt_text = processor.apply_chat_template(
        CAPTION_MESSAGES, add_generation_prompt=True
    )
    prompts = [prompt_text] * len(images)
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items() if torch.is_tensor(v)}
    out = model.generate(
        **inputs, max_new_tokens=80, do_sample=False,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    captions = []
    for i, seq in enumerate(out):
        prompt_len = inputs["attention_mask"][i].sum().item()
        new_tokens = seq[prompt_len:]
        captions.append(
            processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        )
    return captions


def generate_all_captions(dfs, data_dir, save_path):
    """Generate captions for all images, ignoring existing cache."""
    print("Generating captions from scratch...")
    all_paths = sorted(set(p for df in dfs for p in df["image_path"].tolist()))
    captions  = {}
    for i in tqdm(range(0, len(all_paths), CAPTION_BATCH)):
        batch_paths = all_paths[i : i + CAPTION_BATCH]
        images, valid = [], []
        for p in batch_paths:
            try:
                img = Image.open(data_dir / p).convert("RGB").resize(
                    (IMG_SIZE, IMG_SIZE), Image.LANCZOS
                )
                images.append(img)
                valid.append(p)
            except:
                captions[p] = ""
        if images:
            try:
                for p, cap in zip(valid, generate_captions_batch(images)):
                    captions[p] = cap
            except Exception as e:
                for p in valid:
                    captions[p] = ""
                print(f"  Batch failed: {e}")
    with open(save_path, "w") as f:
        json.dump(captions, f)
    print(f"Raw captions saved: {len(captions)} entries -> {save_path}")
    return captions


def is_bad_caption(v):
    v = v.strip()
    if len(v) < 20:
        return True
    neg_starts = [
        "No.", "No ", "None", "Yes.", "Yes ", "Answer:",
        "No visible", "No diagrams", "No diagram",
        "There is no", "There are no",
        "This is a science question image"
    ]
    if any(v.startswith(p) for p in neg_starts):
        return True
    low_info = [
        "Table.", "Fish skeleton.", "Sample A and Sample B.",
        "The map shows the United States.",
        "There is a map of the United States.",
        "The map of the United States"
    ]
    if any(v.startswith(p) for p in low_info):
        return True
    words = v.split()
    if len(words) > 5:
        if len(set(words)) / len(words) < 0.4:
            return True
    return False


def clean_and_save(raw_cache, save_path):
    """Filter low-quality captions and save cleaned cache."""
    cleaned = {k: ("" if is_bad_caption(v) else v.strip())
               for k, v in raw_cache.items()}
    with open(save_path, "w") as f:
        json.dump(cleaned, f)
    good  = sum(1 for v in cleaned.values() if v != "")
    total = len(cleaned)
    print(f"Valid captions: {good}/{total} ({good/total*100:.1f}%)")
    print(f"Filtered out:   {total-good}/{total} ({(total-good)/total*100:.1f}%)")
    return cleaned


# Generate → clean → save
raw_cache     = generate_all_captions([train_df, val_df, test_df], DATA_DIR, CAPTION_RAW)
caption_cache = clean_and_save(raw_cache, CAPTION_CACHE)

print("\nSample valid captions:")
for k, v in [(k, v) for k, v in caption_cache.items() if v != ""][:3]:
    print(f"  {k}: {v[:100]}")

Generating captions from scratch...


  0%|          | 0/162 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
# ready cache
import json
from pathlib import Path

DATA_DIR      = Path("/content/drive/MyDrive/pixels_to_predictions")
CAPTION_CACHE = DATA_DIR / "image_captions_v2_cleaned.json"

with open(CAPTION_CACHE) as f:
    caption_cache = json.load(f)

print(f"已加载 {len(caption_cache)} 条caption")
good = sum(1 for v in caption_cache.values() if v != "")
print(f"有效caption: {good}/{len(caption_cache)} ({good/len(caption_cache)*100:.1f}%)")

已加载 5165 条caption
有效caption: 2290/5165 (44.3%)


In [10]:
!pip install -q "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.7 MB/s eta 0:00:00


In [11]:
# ── 7a. LoRA — Attention + MLP + DoRA ─────────────────────────────────────────


from peft import LoraConfig, get_peft_model, TaskType

if hasattr(model, "peft_config"):
    print("LoRA already injected — skipping")
else:
    target_modules = ["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]

    lora_config = LoraConfig(
        r=4,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        use_dora=True,    # [Tip 2] DoRA: decomposes into magnitude + direction
    )
    model = get_peft_model(model, lora_config)

model.enable_input_require_grads()        # required for VLMs
model.gradient_checkpointing_enable()    # [Tip 9] keep on
model.config.use_cache = False

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        model = model.to(torch.bfloat16)
        print("Using bfloat16")
    else:
        model = model.to(torch.float16)
        print("Using float16")

model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable/1e6:.3f}M  (limit: 5.000M)")
assert trainable <= 5_000_000, f"EXCEEDS 5M LIMIT!"
model.train()
print("✅ LoRA+DoRA setup complete")

Using bfloat16
trainable params: 2,162,688 || all params: 509,644,992 || trainable%: 0.4244
Trainable params: 2.163M  (limit: 5.000M)
✅ LoRA+DoRA setup complete


In [12]:
def collate_fn(batch):
    inputs = processor(
        text=[item["text"] for item in batch],
        images=[item["image"] for item in batch],
        return_tensors="pt",
        padding=True,
    )

    labels = torch.full_like(inputs["input_ids"], -100)
    for i in range(len(batch)):
        real = (inputs["attention_mask"][i] == 1).nonzero(as_tuple=True)[0]
        if len(real) > 0:
            last = real[-1]
            labels[i, last] = inputs["input_ids"][i, last]
    inputs["labels"] = labels
    return inputs

# Sanity check
sample_batch  = [train_ds[i] for i in range(4)]
sample_inputs = collate_fn(sample_batch)
print("Collator check:")
for i in range(4):
    sup    = sample_inputs["labels"][i][sample_inputs["labels"][i] != -100]
    letter = processor.tokenizer.decode(sup).strip()
    exp    = CHOICE_LETTERS[sample_batch[i]["answer"]]
    print(f"  [{i}] label='{letter}' expected='{exp}' {'✅' if letter == exp else '❌'}")


Collator check:
  [0] label='C' expected='C' ✅
  [1] label='A' expected='A' ✅
  [2] label='B' expected='B' ✅
  [3] label='B' expected='B' ✅


In [13]:
# ── 11. Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./smolvlm-science-qa",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    logging_steps=5,
    eval_strategy="no",
    save_strategy="no",

    bf16=True,
    tf32=True,

    dataloader_num_workers=2,
    gradient_checkpointing=True,    # [Tip 9] keep on
    remove_unused_columns=False,
    report_to="none"
)

In [14]:
# ── 12. Start Training ─────────────────────────────────────────────────────────
from transformers import Trainer
import transformers

transformers.utils.logging.set_verbosity_info()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

print("🚀 Starting training...")
trainer.train()

os.makedirs("./output_model", exist_ok=True)
model.save_pretrained("./output_model/science_lora_adapter1")
processor.save_pretrained("./output_model/science_lora_adapter1")
print("✅ Training complete. Weights saved.")

The model is already on multiple devices. Skipping the move to device specified in `args`.
Using auto half precision backend
***** Running training *****
  Num examples = 3,109
  Num Epochs = 3
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 4
  Total optimization steps = 585
  Number of trainable parameters = 2,162,688


🚀 Starting training...


Step,Training Loss
5,2.555700
10,2.267300
15,2.238700
20,1.703300
25,1.157900
30,0.966000
35,0.679000
40,0.679000
45,0.561600
50,0.806100




Training completed. Do not forget to share your model on huggingface.co/models =)


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--HuggingFaceTB--SmolVLM-500M-Instruct/snapshots/a7da5b986cb59b408707209984f360a5f4ad7e47/config.json
vision_config is None, using default vision config
text_config is None, using default text config
Model config Idefics3Config {
  "architectures": [
    "Idefics3ForConditionalGeneration"
  ],
  "dtype": "bfloat16",
  "image_token_id": 49190,
  "model_type": "idefics3",
  "pad_token_id": 128002,
  "scale_factor": 4,
  "text_config": {
    "_attn_implementation_autoset": false,
    "_flash_attn_2_enabled": true,
    "_name_or_path": "None",
    "architectures": [
      "VLlama3ForCausalLM"
    ],
    "attention_bias": false,
    "attention_dropout": 0.0,
    "dtype": "bfloat16",
    "head_dim": 64,
    "hidden_act": "silu",
    "hidden_size": 960,
    "initializer_range": 0.02,
    "intermediate_size": 2560,
    "is

✅ Training complete. Weights saved.


In [15]:
# ── 4a. Multiple-choice scoring (UNCHANGED from your submitted notebook) ────────
import re
from tqdm.auto import tqdm

ANSWER_LETTERS = list("ABCDEFGHIJ")

def get_num_choices(row):
    if "num_choices" in row and pd.notna(row["num_choices"]):
        return int(row["num_choices"])
    return len(row["choices"])

@torch.inference_mode()
def score_candidate_letter(model, processor, image, prompt, candidate_letter: str) -> float:
    """
    Compute NLL for one candidate answer letter conditioned on image + prompt.
    Lower score = model considers that answer more likely.
    """
    answer_text = " " + candidate_letter
    full_text = prompt + answer_text

    full_inputs = processor(
        text=[full_text],
        images=[image],
        return_tensors="pt",
    )
    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    full_inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in full_inputs.items()}
    prompt_len = prompt_inputs["input_ids"].shape[1]

    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100

    outputs = model(**full_inputs, labels=labels)
    return float(outputs.loss.detach().cpu())

@torch.inference_mode()
def predict_one(row, split: str = "val") -> int:
    image  = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)
    n_choices = get_num_choices(row)

    scores = [
        score_candidate_letter(model, processor, image, prompt, ANSWER_LETTERS[i])
        for i in range(n_choices)
    ]
    return int(np.argmin(scores))

In [16]:
# ── Evaluate on full validation set ───────────────────────────────────────────
model.eval()

val_preds = []
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    val_preds.append(predict_one(row, split="val"))

val_true = val_df["answer"].astype(int).tolist()
val_acc  = np.mean([p == y for p, y in zip(val_preds, val_true)])
print(f"Val accuracy: {val_acc:.4f} ({sum(p==y for p,y in zip(val_preds,val_true))}/{len(val_true)})")

pd.DataFrame({
    "id":      val_df["id"].values,
    "true":    val_true,
    "pred":    val_preds,
    "correct": [p == y for p, y in zip(val_preds, val_true)],
}).head(20)

  0%|          | 0/1048 [00:00<?, ?it/s]

Val accuracy: 0.7767 (814/1048)


,id,true,pred,correct
0,val_00671,0,1,False
1,val_04111,1,0,False
2,val_02022,3,0,False
3,val_01237,0,1,False
4,val_03458,4,1,False
5,val_04064,3,3,True
6,val_03959,4,1,False
7,val_03289,2,0,False
8,val_03452,3,2,False
9,val_01021,0,2,False


In [17]:
# ── 6a. Predict test set and save submission.csv ────────────────────────────────
test_preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    test_preds.append(predict_one(row, split="test"))

submission = pd.DataFrame({
    "id":     test_df["id"].values,
    "answer": np.array(test_preds, dtype=int),
})

# Strict format checks (identical to submitted notebook)
assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(test_df)
assert submission["id"].tolist() == test_df["id"].tolist()
assert submission["answer"].notna().all()
assert pd.api.types.is_integer_dtype(submission["answer"])

for i, row in test_df.reset_index(drop=True).iterrows():
    assert 0 <= int(submission.loc[i, "answer"]) < get_num_choices(row)

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

  0%|          | 0/1008 [00:00<?, ?it/s]

Exception ignored in: <generator object tqdm_notebook.__iter__ at 0x7e4b6b44dc40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 250, in __iter__
    yield from it
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1196, in __iter__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
  File "/usr/local/lib/python3.12/dist-packages/tqdm/notebook.py", line 171, in display
    rtext.value = right
    ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/traitlets/traitlets.py", line 729, in __set__
    self.set(obj, value)
  File "/usr/local/lib/python3.12/dist-packages/traitlets/traitlets.py", line 718, in set
    obj._notify_trait(self.name, old_value, new_value)
  File "/usr/local/lib/python3.12/dist-packages/traitlets/traitlets.py", line 1501, in _notify_trait
    self.notify_change(
  File "/usr/local

KeyboardInterrupt: 

In [ ]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/final_project.ipynb"

with open(notebook_path, "r") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("Fixed.")

Fixed.
